# AstroHealth: train the CDSS symptom model (Colab)

Trains the differential-diagnosis MLP on the **synthetic** dataset made by `generate-synthetic-data.js`, then exports it as a **TensorFlow.js** model for your React app.

**How to run**
1. Runtime > Run all (a GPU is not needed).
2. When asked, upload `cdss-synthetic.json` (from `data/synthetic/`).
3. At the end, a `cdss-mlp.zip` downloads. Unzip it into `public/models/cdss-mlp/` in your project.

> The data is synthetic and rule-derived, so high accuracy here shows the pipeline works, not that the model is clinically valid. Label it as a demo / decision-support in your app.

In [ ]:
import json, os, shutil, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)
print("TensorFlow", tf.__version__)

## 1. Load the data

In [ ]:
DATA_FILE = "cdss-synthetic.json"
if not os.path.exists(DATA_FILE):
    try:
        from google.colab import files
        print("Select cdss-synthetic.json ...")
        uploaded = files.upload()
        DATA_FILE = next(iter(uploaded))
    except ImportError:
        raise FileNotFoundError("Put cdss-synthetic.json next to this notebook.")

with open(DATA_FILE) as f:
    D = json.load(f)

feature_names = D["featureNames"]
class_names = D["classNames"]

def get(split):
    s = D["splits"][split]
    return np.array(s["X"], dtype="float32"), np.array(s["y"], dtype="int32")

X_train, y_train = get("train")
X_val,   y_val   = get("val")
X_test,  y_test  = get("test")

print("features:", X_train.shape[1], "| classes:", len(class_names))
print("train / val / test:", len(y_train), len(y_val), len(y_test))
print(D["meta"]["disclaimer"])
pd.Series(y_train).map(dict(enumerate(class_names))).value_counts()

## 2. Quick baselines
Simple models first. If a basic model already scores high, the synthetic classes are easy to separate, which is useful to know before you read too much into the neural network's score.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

lr = LogisticRegression(max_iter=2000).fit(X_train, y_train)
gb = HistGradientBoostingClassifier(random_state=SEED).fit(X_train, y_train)
print(f"Logistic regression  val acc: {accuracy_score(y_val, lr.predict(X_val)):.3f}")
print(f"Gradient boosting    val acc: {accuracy_score(y_val, gb.predict(X_val)):.3f}")

## 3. Train the neural network
Small search over three sizes. The final layer is softmax, so outputs are the probabilities your UI shows. Class weights stop the smaller classes from being ignored. Only `Dense` and `Dropout` layers are used so the TF.js export in step 6 stays simple.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

n_features, n_classes = X_train.shape[1], len(class_names)
cw = compute_class_weight("balanced", classes=np.arange(n_classes), y=y_train)
class_weight = dict(enumerate(cw))

def build_model(hidden, dropout=0.2, lr=1e-3):
    layers = [keras.Input(shape=(n_features,))]
    for h in hidden:
        layers += [keras.layers.Dense(h, activation="relu"), keras.layers.Dropout(dropout)]
    layers += [keras.layers.Dense(n_classes, activation="softmax")]
    m = keras.Sequential(layers)
    m.compile(optimizer=keras.optimizers.Adam(lr),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

callbacks = lambda: [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-5),
]

results = {}
for hidden in [(64, 32), (128, 64, 32), (256, 128, 64)]:
    m = build_model(hidden)
    h = m.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=100,
              batch_size=64, class_weight=class_weight, callbacks=callbacks(), verbose=0)
    va = max(h.history["val_accuracy"])
    results[hidden] = (va, m, h)
    print(f"hidden={hidden}: best val acc {va:.4f}  ({len(h.history['loss'])} epochs)")

best_hidden = max(results, key=lambda k: results[k][0])
_, model, history = results[best_hidden]
print("\nChosen:", best_hidden)
model.summary()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(history.history["loss"], label="train"); ax[0].plot(history.history["val_loss"], label="val")
ax[0].set_title("Loss"); ax[0].legend()
ax[1].plot(history.history["accuracy"], label="train"); ax[1].plot(history.history["val_accuracy"], label="val")
ax[1].set_title("Accuracy"); ax[1].legend()
plt.tight_layout(); plt.show()

## 4. Evaluate on the test set
The test split was never used for training or model selection. Look at the per-class numbers and the confusion matrix, not only overall accuracy.

In [ ]:
probs = model.predict(X_test, verbose=0)
pred = probs.argmax(1)
test_acc = accuracy_score(y_test, pred)
print(f"TEST accuracy: {test_acc:.4f}\n")
print(classification_report(y_test, pred, target_names=class_names, digits=3))

cm = confusion_matrix(y_test, pred)
fig, ax = plt.subplots(figsize=(7, 6))
ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(n_classes)); ax.set_xticklabels(class_names, rotation=45, ha="right")
ax.set_yticks(range(n_classes)); ax.set_yticklabels(class_names)
for i in range(n_classes):
    for j in range(n_classes):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("Confusion matrix (test)")
plt.tight_layout(); plt.show()

## 5. Robustness check
Real sensor readings are noisier than the training data. This adds noise to the test inputs and shows how accuracy drops. A model that collapses quickly is a warning sign.

In [ ]:
rng = np.random.default_rng(SEED)
for sd in [0.0, 0.03, 0.06, 0.10, 0.15]:
    Xn = np.clip(X_test + rng.normal(0, sd, X_test.shape), 0, 1).astype("float32")
    print(f"noise sd={sd:.2f}  accuracy={accuracy_score(y_test, model.predict(Xn, verbose=0).argmax(1)):.3f}")

## 6. Export for TensorFlow.js
This writes the TF.js layers-model format directly (`model.json` + `weights.bin`), so it does not depend on the `tensorflowjs` converter package, which often breaks on newer Keras versions. It then re-reads the files and checks they reproduce the Keras predictions.

In [ ]:
def export_tfjs(model, out_dir, class_names, feature_names, test_acc):
    os.makedirs(out_dir, exist_ok=True)
    layers_json, specs, chunks = [], [], []
    d_i = dr_i = 0
    first = True
    for layer in model.layers:
        if isinstance(layer, keras.layers.Dense):
            d_i += 1
            name = f"dense_Dense{d_i}"
            cfg = {
                "units": layer.units, "activation": layer.activation.__name__, "use_bias": True,
                "kernel_initializer": {"class_name": "VarianceScaling",
                    "config": {"scale": 1, "mode": "fan_avg", "distribution": "normal", "seed": None}},
                "bias_initializer": {"class_name": "Zeros", "config": {}},
                "kernel_regularizer": None, "bias_regularizer": None, "activity_regularizer": None,
                "kernel_constraint": None, "bias_constraint": None,
                "name": name, "trainable": True,
            }
            if first:
                cfg["batch_input_shape"] = [None, n_features]; cfg["dtype"] = "float32"; first = False
            layers_json.append({"class_name": "Dense", "config": cfg})
            kernel, bias = [w.astype("<f4") for w in layer.get_weights()]
            specs += [{"name": f"{name}/kernel", "shape": list(kernel.shape), "dtype": "float32"},
                      {"name": f"{name}/bias",   "shape": list(bias.shape),   "dtype": "float32"}]
            chunks += [kernel.tobytes(), bias.tobytes()]
        elif isinstance(layer, keras.layers.Dropout):
            dr_i += 1
            layers_json.append({"class_name": "Dropout", "config": {
                "rate": float(layer.rate), "noise_shape": None, "seed": None,
                "name": f"dropout_Dropout{dr_i}", "trainable": True}})
        else:
            raise ValueError(f"Unsupported layer {layer}")

    model_json = {
        "modelTopology": {"class_name": "Sequential",
                          "config": {"name": "cdss_mlp", "layers": layers_json},
                          "keras_version": "2.15.0", "backend": "tensorflow"},
        "format": "layers-model", "generatedBy": "astro-health colab notebook", "convertedBy": None,
        "weightsManifest": [{"paths": ["weights.bin"], "weights": specs}],
    }
    with open(f"{out_dir}/model.json", "w") as f: json.dump(model_json, f)
    with open(f"{out_dir}/weights.bin", "wb") as f: f.write(b"".join(chunks))
    with open(f"{out_dir}/meta.json", "w") as f:
        json.dump({"classNames": class_names, "featureNames": feature_names,
                   "syntheticData": True, "testAccuracy": float(test_acc)}, f, indent=2)

def verify_export(out_dir, model, X):
    mj = json.load(open(f"{out_dir}/model.json"))
    raw = np.fromfile(f"{out_dir}/weights.bin", dtype="<f4")
    specs, off, params = mj["weightsManifest"][0]["weights"], 0, {}
    for s in specs:
        n = int(np.prod(s["shape"])); params[s["name"]] = raw[off:off + n].reshape(s["shape"]); off += n
    a, i = X, 1
    while f"dense_Dense{i}/kernel" in params:
        a = a @ params[f"dense_Dense{i}/kernel"] + params[f"dense_Dense{i}/bias"]
        last = f"dense_Dense{i+1}/kernel" not in params
        if last:
            e = np.exp(a - a.max(1, keepdims=True)); a = e / e.sum(1, keepdims=True)
        else:
            a = np.maximum(a, 0)
        i += 1
    ok = np.allclose(a, model.predict(X, verbose=0), atol=1e-5)
    print("Export check (re-read files == Keras output):", "PASS" if ok else "FAIL")
    assert ok

OUT = "cdss-mlp"
export_tfjs(model, OUT, class_names, feature_names, test_acc)
verify_export(OUT, model, X_test[:200])
print(os.listdir(OUT), f"weights.bin = {os.path.getsize(OUT + '/weights.bin') / 1024:.1f} KB")

In [ ]:
shutil.make_archive("cdss-mlp", "zip", ".", OUT)
try:
    from google.colab import files
    files.download("cdss-mlp.zip")
except ImportError:
    print("Created cdss-mlp.zip")

## 7. Use it in the React app
1. Unzip `cdss-mlp.zip` into `public/models/cdss-mlp/` (it contains `model.json`, `weights.bin`, `meta.json`).
2. Load it in your TF.js worker:

```js
import * as tf from '@tensorflow/tfjs';
import { encode } from '../ml/features.js';

const model = await tf.loadLayersModel('/models/cdss-mlp/model.json');
const meta = await (await fetch('/models/cdss-mlp/meta.json')).json();

export function predict(record) {            // { symptoms: {...}, context: {...} }
  return tf.tidy(() => {
    const x = tf.tensor2d([encode(record)]);
    const p = model.predict(x).dataSync();
    return meta.classNames.map((name, i) => ({ name, probability: p[i] }))
                          .sort((a, b) => b.probability - a.probability);
  });
}
```

The feature order in `features.js` must match `featureNames` in `meta.json`. Keep the rule engine in front of the model so red-flag emergencies are never overridden by it.